# Genre classification of Steam games with ICL: ablation study
Model: `mistral-small-3-2-24b-instruct-2506` via KI:connect, no fine-tuning.
Ablation: ICL technique (zero-shot, few-shot random, few-shot kNN) × description scope (short, about).

**How to run:** set `TECHNIQUE` and `RUN` in the configuration and run all cells. Existing raw outputs and embeddings are loaded, not re-queried.
- "Per-Technique Evaluation" evaluates the technique selected in the configuration (preliminary, B = 2,000, uncorrected).
- "Joint evaluation" at the end is independent of `TECHNIQUE`/`RUN` and produces all reported results (B = 10,000, Bonferroni-corrected).

# Setup & Configuration

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import time

from openai import OpenAI, RateLimitError, APITimeoutError, APIConnectionError, InternalServerError
import os
from dotenv import load_dotenv
import json
import re
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (precision_score, recall_score, f1_score, hamming_loss, accuracy_score, classification_report)
from collections import Counter

pd.set_option("display.max_colwidth", None) # global setting for all tables in this notebook

In [2]:
# Paths
DATA_DIR = Path("Data")
RAW_DIR = Path("results/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Seeds
SPLIT_SEED = 42 # seed of the train/val/test split (in data_preprocessing.ipynb)
RANDOM_SEED = 27 # used for pilot samples, demonstration pool, demo selection/order, bootstrap

# Run selection
TECHNIQUE = "few_shot_knn" # different ICL techniques which will be used in the experiment: zero_shot | few_shot_random | few_shot_knn
RUN = "main" # pilot | main

# Model
MODEL = "mistral-small-3-2-24b-instruct-2506" # instruct model without built-in reasoning, so no hidden chain-of-thought confounds the comparison of ICL techniques
MODEL_TEMP = 0 # greedy decoding; server-side outputs may still vary slightly
MAX_TOKENS = 100 # upper limit for output token lenght (longest observed completion: 36 tokens)
RESPONSE_FORMAT = {"type": "json_object"} # enforces a JSON object as output of the model

# Labels and input conditions
CONTENT_GENRES = ["Action", "Adventure", "Massively Multiplayer", "Racing", "RPG", "Simulation", "Sports", "Strategy"] # all valid content genres (non-content labels were removed in preprocessing)
DESCRIPTION_LEVELS = {
    "short": "short_description", # Steam "Short Description" (a short description of the game)
    "about": "about_the_game", # Steam "About the Game" (a long description of the game)
}
CONDITIONS = { # variant -> (text column, field label in the prompt)
    **{v: (c, "Description") for v, c in DESCRIPTION_LEVELS.items()}, 
    "name_only": ("name", "Game title"), # control: title only (zero-shot)
}
FEWSHOT_CONDITIONS = {v: CONDITIONS[v] for v in DESCRIPTION_LEVELS} # no name_only control
ORDER = ["short", "about", "name_only", "majority"] # display order for generated tables

# Few-shot
K = 4 # demonstrations/samples shown per prompt in few-shot settings
POOL_SIZE = 5000 # demonstration pool from train set, shared by all few-shot techniques
FEWSHOT_PILOT_N = 20 # pilot games from the validation set

# kNN retrieval
EMBED_MODEL = "qwen3-embedding-8b" # used only for kNN retrieval
EMBED_BATCH = 25 # texts per embeddings request

# Data
Preprocessed splits from `data_preprocessing.ipynb` (iterative multi-label stratification, split seed 42):
- `train_set` (57,799 games): source of the zero-shot pilot sample (n = 40) and of the few-shot demonstration pool (5,000 games). No training takes place.
- `val_set` (1,003 games): few-shot pilots (n = 20).
- `test_set` (1,009 games): evaluation set for all main runs.

Columns: `name`, `short_description`, `about_the_game`, `genres` (gold labels; stored as NumPy arrays and converted to lists below), `leak_short_description` / `leak_about_the_game` (True if the description names one of its gold genres verbatim). `detailed_description` was dropped in preprocessing because it is nearly identical to `about_the_game`.

In [3]:
train_df, val_df, test_df = (pd.read_parquet(DATA_DIR / f"{s}_set.parquet") for s in ("train", "val", "test"))
print({name: len(df) for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]})

{'train': 57799, 'val': 1003, 'test': 1009}


In [4]:
# ensure that the ‘genres’ column contains only lists, so that the following code does not crash
for df in (train_df, val_df, test_df):
    df["genres"] = df["genres"].apply(list)

In [5]:
# Demonstration pool: fixed random subset of the training set, shared by both few-shot techniques random and kNN
pool_df = train_df.sample(POOL_SIZE, random_state=RANDOM_SEED).reset_index(drop=True)
for col in DESCRIPTION_LEVELS.values():
    assert pool_df[col].fillna("").str.strip().str.len().gt(0).all(), f"empty texts in {col}"
print("pool size:", len(pool_df))

pool size: 5000


# Model & Prompting

In [6]:
load_dotenv()
client = OpenAI(api_key=os.environ["KICONNECT_API_KEY"], base_url="https://chat.kiconnect.nrw/api/v1") # KI:connect provides an OpenAI-compatible endpoint (API key is read from an .env file)
print([m.id for m in client.models.list().data])

['mistral-small-3-2-24b-instruct-2506', 'openai-gpt-oss-120b', 'mistral-small-4-119b-2603', 'qwen3-8-27b', 'qwen3-embedding-8b', 'e5-mistral-7b-instruct']


## Prompt and model call
- The system prompt is identical for all conditions; only the user field label differs (`Description` vs. `Game title`).
- "Use only genres from the list." was added after the first pilot (n = 8) because of out-of-list labels. The prompt is frozen from here on.
- Each call returns the raw output, token usage and `finish_reason` (`"length"` = truncated).

In [7]:
SYSTEM_PROMPT = (
    "You are a classifier for video game genres.\n"
    f"Allowed genres: {', '.join(CONTENT_GENRES)}.\n"
    "Assign every genre from the list that applies to the game, using only the information given.\n"
    "Use only genres from the list.\n"
    'Respond only with JSON in the form {"genres": ["..."]}. No further text.'
)

RETRYABLE = (RateLimitError, APITimeoutError, APIConnectionError, InternalServerError)

def make_demos(demo_df, col):
    """Converts pool rows into few-shot demonstrations.

    Args:
        demo_df: Pool rows selected as demonstrations.
        col: Name of the description column to use as demonstration text.

    Returns:
        List of (text, genres) tuples; genres are ordered as in CONTENT_GENRES.
    """
    
    return [(r[col], [g for g in CONTENT_GENRES if g in r["genres"]]) for _, r in demo_df.iterrows()]

def build_messages(text, field="Description", demos=None):
    """Builds the chat messages for one classification request.

    Args:
        text: Input text of the game to classify.
        field: Field label in the user turn ("Description" or "Game title").
        demos: Optional list of (text, genres) tuples, inserted as
            user/assistant turns before the query.

    Returns:
        List of message dicts in OpenAI chat format.
    """
    
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]
    for demo_text, demo_genres in (demos or []):
        msgs.append({"role": "user", "content": f"{field}:\n{demo_text}"})
        msgs.append({"role": "assistant", "content": json.dumps({"genres": demo_genres})})
    msgs.append({"role": "user", "content": f"{field}:\n{text}"})
    return msgs

def classify(text, field="Description", demos=None):
    """Sends one classification request to the model.

    Args:
        text: Input text of the game to classify.
        field: Field label in the user turn.
        demos: Optional few-shot demonstrations, see build_messages.

    Returns:
        Dict with the raw output, prompt and completion tokens and finish_reason.
    """
    
    resp = client.chat.completions.create(
        model=MODEL,
        messages=build_messages(text, field, demos),
        temperature=MODEL_TEMP,
        max_tokens=MAX_TOKENS,
        response_format=RESPONSE_FORMAT,
    )
    
    return {
        "raw": resp.choices[0].message.content,
        "prompt_tokens": resp.usage.prompt_tokens,
        "finish_reason": resp.choices[0].finish_reason,
        "completion_tokens": resp.usage.completion_tokens,
    }

def classify_with_retry(text, field, demos=None, max_tries=5):
    """Calls classify and retries transient API errors with exponential backoff.

    Args:
        text: Input text of the game to classify.
        field: Field label in the user turn.
        demos: Optional few-shot demonstrations, see build_messages.
        max_tries: Maximum number of attempts.

    Returns:
        Result dict of classify.

    Raises:
        RuntimeError: If all attempts fail.
    """
    
    for attempt in range(max_tries):
        try:
            return classify(text, field, demos)
        except RETRYABLE as e:
            wait = 2 ** attempt
            print(f"{type(e).__name__}, retry in {wait}s")
            time.sleep(wait)
    raise RuntimeError("Max retries reached")

def run_conditions(games, out_path, conditions, selector=None):
    """Queries all games under all conditions and appends results to a JSONL file.

    Pairs (appid, variant) already present in out_path are skipped, so an
    interrupted run can be resumed.

    Args:
        games: Games to classify.
        out_path: Path of the JSONL file with the raw outputs.
        conditions: Mapping variant -> (text column, field label).
        selector: Optional function (game, variant) -> demonstration rows;
            None for zero-shot.

    Returns:
        DataFrame with all rows of out_path.
    """
    
    done = set()
    if out_path.exists():
        with out_path.open() as f:
            done = {(r["appid"], r["variant"]) for r in map(json.loads, f)}
    todo = [(g, v) for _, g in games.iterrows() for v in conditions
            if (int(g["appid"]), v) not in done]
    print(f"{len(done)} done, {len(todo)} to go")

    t0 = time.time()
    with out_path.open("a") as f:
        for i, (game, variant) in enumerate(todo, 1):
            col, field = conditions[variant]
            demos, demo_ids = None, []
            if selector is not None:
                demo_df = selector(game, variant)
                demos = make_demos(demo_df, col)
                demo_ids = demo_df["appid"].astype(int).tolist()
            out = classify_with_retry(game[col], field, demos)
            f.write(json.dumps({
                "appid": int(game["appid"]), "variant": variant,
                "gold": list(game["genres"]), "demo_ids": demo_ids, **out,
            }) + "\n")
            f.flush()
            if i % 100 == 0:
                print(f"{i}/{len(todo)} | {(time.time() - t0) / i:.2f} s/call")
    return pd.read_json(out_path, lines=True)

def select_random(game, variant):
    """Selects K random pool games, seeded per query game.

    The seed depends only on the game, so short and about receive the
    same demonstration games.

    Args:
        game: Query game (row with appid).
        variant: Description level; unused, kept for a uniform selector interface.

    Returns:
        K rows of pool_df.
    """
    
    rng = np.random.default_rng([RANDOM_SEED, int(game["appid"])])
    return pool_df.iloc[rng.choice(len(pool_df), K, replace=False)]

SELECTORS = {"zero_shot": None, "few_shot_random": select_random}

# Embeddings and kNN retrieval

**Prior timing test: embed 50 pool texts per level and extrapolate**
```
timing_sample = pool_df.sample(50, random_state=RANDOM_SEED)
n_total = len(pool_df) + len(test_df) + FEWSHOT_PILOT_N   # pool + test + pilot games

for level, col in DESCRIPTION_LEVELS.items():
    texts = timing_sample[col].tolist()
    t0 = time.time()
    for i in range(0, len(texts), EMBED_BATCH):
        client.embeddings.create(model=EMBED_MODEL, input=texts[i:i + EMBED_BATCH])
    per_text = (time.time() - t0) / len(texts)
    print(f"{level}: {per_text:.3f} s/Text --> ca. {per_text * n_total / 60:.0f} min "
          f"for {n_total} texts")
```

**Output:**
`short: 0.009 s/text --> ca. 1 min for 6029 texts`
`about: 0.022 s/text → ca. 2 min for 6029 texts`

In [8]:
EMB_DIR = Path("results/embeddings")
EMB_DIR.mkdir(parents=True, exist_ok=True)

def embed_texts(texts):
    """Embeds texts in batches with retries.

    Args:
        texts: List of strings.

    Returns:
        float32 array of shape (len(texts), dim), L2-normalised.

    Raises:
        RuntimeError: If a batch fails after all retries.
    """
    
    out = []
    for i in range(0, len(texts), EMBED_BATCH):
        batch = texts[i:i + EMBED_BATCH]
        for attempt in range(5):
            try:
                r = client.embeddings.create(model=EMBED_MODEL, input=batch)
                break
            except RETRYABLE:
                time.sleep(2 ** attempt)
        else:
            raise RuntimeError("Embedding request failed")
        out.extend(d.embedding for d in sorted(r.data, key=lambda d: d.index))
    X = np.asarray(out, dtype=np.float32)
    return X / np.linalg.norm(X, axis=1, keepdims=True)

def get_embeddings(name, df, level):
    """Loads cached embeddings or computes and caches them.

    The cache is used only if its stored appids match df exactly.

    Args:
        name: Set name used in the file name ("pool", "val", "test").
        df: Games to embed.
        level: Description level ("short" or "about").

    Returns:
        Embedding matrix with one row per game in df.
    """
    
    path, ids_path = EMB_DIR / f"{name}_{level}.npy", EMB_DIR / f"{name}_appids.npy"
    ids = df["appid"].to_numpy()
    if path.exists() and ids_path.exists() and np.array_equal(np.load(ids_path), ids):
        return np.load(path)
    X = embed_texts(df[DESCRIPTION_LEVELS[level]].tolist())
    np.save(path, X)
    np.save(ids_path, ids)
    return X

SETS = [("pool", pool_df), ("val", val_df), ("test", test_df)]
EMB = {(name, level): get_embeddings(name, df, level) for name, df in SETS for level in DESCRIPTION_LEVELS}

# appid of a query game -> (set name, row index in its embedding matrix)
QUERY_ROWS = {int(a): (name, i) for name, df in SETS[1:] for i, a in enumerate(df["appid"])}

def select_knn(game, variant):
    """Selects the K pool games most similar to the query (cosine similarity).

    Similarity is computed on the same description level as the query.
    The order of the K neighbours is shuffled with a per-game seed.

    Args:
        game: Query game from val or test (row with appid).
        variant: Description level ("short" or "about").

    Returns:
        K rows of pool_df.
    """
    
    name, i = QUERY_ROWS[int(game["appid"])]
    sims = EMB[("pool", variant)] @ EMB[(name, variant)][i]
    top = np.argsort(-sims)[:K]
    order = np.random.default_rng([RANDOM_SEED, int(game["appid"])]).permutation(top)
    return pool_df.iloc[order]

SELECTORS["few_shot_knn"] = select_knn
print({k: v.shape for k, v in EMB.items()})

{('pool', 'short'): (5000, 4096), ('pool', 'about'): (5000, 4096), ('val', 'short'): (1003, 4096), ('val', 'about'): (1003, 4096), ('test', 'short'): (1009, 4096), ('test', 'about'): (1009, 4096)}


In [9]:
# check if meaningful and similar neighbours have been found

for _, g in val_df.sample(3, random_state=RANDOM_SEED).iterrows():
    name, i = QUERY_ROWS[int(g["appid"])]
    for level in DESCRIPTION_LEVELS:
        sims = EMB[("pool", level)] @ EMB[(name, level)][i]
        print(f"\n[{level}] {g['name']} | Gold: {g['genres']}")
        for j in np.argsort(-sims)[:K]:
            print(f"   {sims[j]:.3f}  {pool_df.iloc[j]['name']} | {pool_df.iloc[j]['genres']}")


[short] Seven Bullets Horror 7発の弾丸ホラー | Gold: ['Action']
   0.679  Into The Depths | ['Action', 'Adventure']
   0.678  Urban riots | ['Action', 'Adventure']
   0.671  Dead Garden Zero | ['Action', 'Strategy']
   0.659  深空七号 Deep Space 7 | ['Action']

[about] Seven Bullets Horror 7発の弾丸ホラー | Gold: ['Action']
   0.698  Labyrinth of Wild Abyss: LayeRedux | ['Action']
   0.662  幽霊写真-GhostShot | ['Action']
   0.641  Urban riots | ['Action', 'Adventure']
   0.640  A Killer's Sorrow | ['Action']

[short] Toska | Gold: ['Adventure']
   0.756  Walking In The Abyss - Definitive Edition | ['Action']
   0.703  Lotharus - Bacon, Ale & Repeat | ['Adventure', 'RPG']
   0.700  sun-rise.exe | ['Adventure']
   0.690  Secrets of the Past: Mother's Diary | ['Adventure']

[about] Toska | Gold: ['Adventure']
   0.615  Strange | ['Adventure', 'Simulation']
   0.606  Trade Post Forest | ['Adventure']
   0.597  Secrets of the Past: Mother's Diary | ['Adventure']
   0.575  Foxen Island | ['Adventure']

[short] 

# Runs
**Pilots.** Purpose: test the pipeline, estimate runtime and settle open decisions (e.g. the synonym table) before the main runs, **not** to tune the prompt.
- **Zero-shot pilot:** 40 games from `train` (short, about, name_only). Outputs are frozen in `pilot_descriptions.jsonl` and `pilot_name_only.jsonl` and loaded instead of re-querying the API, since outputs are not fully deterministic.
- **Few-shot pilots:** the same 20 games from `val` for Random and kNN (`RUN = "pilot"`), written to `results/raw/{TECHNIQUE}_val_pilot.jsonl`.

**Main runs** (`RUN = "main"`): all 1,009 test games, written to `results/raw/{TECHNIQUE}_test.jsonl`. Finished (game, variant) pairs are skipped, so an interrupted run can be resumed.

In [10]:
pilot_games = train_df.sample(40, random_state=RANDOM_SEED)

In [11]:
P_DESC = Path("pilot_descriptions.jsonl")

if P_DESC.exists():
    pilot_desc_res = pd.read_json(P_DESC, lines=True)
    print(f"Loaded: {P_DESC} ({len(pilot_desc_res)} rows)")
else:
    t0 = time.time()
    rows_desc = []
    
    for _, game in pilot_games.iterrows():
        for desc_level, desc_col in DESCRIPTION_LEVELS.items():
            out = classify(game[desc_col], field="Description")
            rows_desc.append({
                "appid": game["appid"],
                "variant": desc_level,
                "gold": game["genres"],
                "raw": out["raw"],
                "prompt_tokens": out["prompt_tokens"],
                "completion_tokens": out["completion_tokens"],
                "finish_reason": out["finish_reason"],
            })

    pilot_desc_res = pd.DataFrame(rows_desc)
    pilot_desc_res.to_json(P_DESC, orient="records", lines=True)
    print(f"Newly created and saved: {P_DESC}")
    print(f"{(time.time() - t0) / len(rows_desc):.2f} s per call")

Loaded: pilot_descriptions.jsonl (80 rows)


In [12]:
# Short control experiment 'title only': measures prior knowledge of the model on games to assess the extent to which the appearance of game titles in the description influences performance. 
# Does the model simply reflect its knowledge of the game, or is it actually attempting to deduce the genre from the description?
P_NAME = Path("pilot_name_only.jsonl")

if P_NAME.exists():
    pilot_name_res = pd.read_json(P_NAME, lines=True)
    print(f"Loaded: {P_NAME} ({len(pilot_name_res)} rows)")
else:
    rows_name = []
    
    for _, game in pilot_games.iterrows():
        out = classify(game["name"], field="Game title")
        rows_name.append({
            "appid": game["appid"],
            "variant": "name_only",
            "gold": game["genres"],
            "raw": out["raw"],
            "prompt_tokens": out["prompt_tokens"],
            "completion_tokens": out["completion_tokens"],
            "finish_reason": out["finish_reason"],
        })

    pilot_name_res = pd.DataFrame(rows_name)
    pilot_name_res.to_json(P_NAME, orient="records", lines=True)
    print(f"Newly generated and saved: {P_NAME}")

Loaded: pilot_name_only.jsonl (40 rows)


In [13]:
# Show one complete prompt for the current technique before running
g = val_df.iloc[0]
sel = SELECTORS[TECHNIQUE]
demos = make_demos(sel(g, "short"), "short_description") if sel else None
for m in build_messages(g["short_description"], "Description", demos):
    print(f"--- {m['role']} ---\n{m['content']}\n")

--- system ---
You are a classifier for video game genres.
Allowed genres: Action, Adventure, Massively Multiplayer, Racing, RPG, Simulation, Sports, Strategy.
Assign every genre from the list that applies to the game, using only the information given.
Use only genres from the list.
Respond only with JSON in the form {"genres": ["..."]}. No further text.

--- user ---
Description:
Play as Sam, freshly arrived in Paris's outskirts from his native countryside, who finds himself embroiled in a rollercoaster ride of adventures. Discover this mysterious world and meet colorful characters who will help you become familiar with local customs.

--- assistant ---
{"genres": ["RPG"]}

--- user ---
Description:
Farmer ? Hunter ? Trader ? Everything is under your control in the green valley. Sow your field, feed your animals, and go hunting in the forest. Don't forget to develop your products and sell them for a much higher amount of gold!

--- assistant ---
{"genres": ["Simulation"]}

--- user --

In [14]:
if TECHNIQUE == "zero_shot" and RUN == "pilot":
    res_all = pd.concat([pilot_desc_res, pilot_name_res], ignore_index=True)
    games = pilot_games
    EVAL_SPLIT = "train (pilot sample)"
else:
    if RUN == "pilot":
        games = val_df.sample(FEWSHOT_PILOT_N, random_state=RANDOM_SEED)
        EVAL_SPLIT, split_tag = "val (pilot sample)", "val_pilot"
    else:
        games = test_df
        EVAL_SPLIT, split_tag = "test", "test"
    conds = CONDITIONS if TECHNIQUE == "zero_shot" else FEWSHOT_CONDITIONS
    res_all = run_conditions(games, RAW_DIR / f"{TECHNIQUE}_{split_tag}.jsonl",
                             conds, SELECTORS[TECHNIQUE])

assert set(res_all["appid"]) == set(games["appid"]), "res_all does not cover exactly the evaluated games"

# Checks on the raw data
print(res_all["variant"].value_counts())
print(res_all["finish_reason"].value_counts())
print(res_all.groupby("variant")[["prompt_tokens", "completion_tokens"]].agg(["mean", "max"]))
display(res_all.sort_values(["appid", "variant"])[["appid", "variant", "gold", "raw"]])

2018 done, 0 to go
variant
short    1009
about    1009
Name: count, dtype: int64
finish_reason
stop    2018
Name: count, dtype: int64
        prompt_tokens        completion_tokens    
                 mean    max              mean max
variant                                           
about     1618.642220  15807         12.341923  32
short      340.895937    469         11.938553  29


,appid,variant,gold,raw
349,39660,about,"[Simulation, Strategy]","{""genres"": [""Simulation""]}"
348,39660,short,"[Simulation, Strategy]","{""genres"": [""RPG"", ""Simulation"", ""Strategy""]}"
217,57620,about,[Strategy],"{""genres"": [""Simulation"", ""Strategy""]}"
216,57620,short,[Strategy],"{""genres"": [""RPG"", ""Simulation"", ""Strategy""]}"
221,63000,about,"[Action, Strategy]","{""genres"": [""Action"", ""Massively Multiplayer"", ""Strategy""]}"
...,...,...,...,...
1384,3507030,short,[Action],"{""genres"": [""Action""]}"
1477,3509390,about,[Action],"{""genres"": [""Simulation""]}"
1476,3509390,short,[Action],"{""genres"": [""Simulation""]}"
1381,3524700,about,"[Action, Adventure]","{""genres"": [""Adventure""]}"


In [15]:
print("Duplicate entries:", res_all.duplicated(["appid", "variant"]).sum()) # expecting '0'
print("appid is unique in the evaluated games:", games["appid"].is_unique) # expecting 'True'
print("Same number of games per variant:", res_all.groupby("variant")["appid"].apply(frozenset).nunique() == 1) # expecting 'True'

Duplicate entries: 0
appid is unique in the evaluated games: True
Same number of games per variant: True


In [16]:
if TECHNIQUE != "zero_shot":
    # demonstrations never come from the evaluated games
    assert not set(res_all["demo_ids"].explode().dropna()) & set(games["appid"])
    print("No evaluated games among the demonstrations")

if TECHNIQUE == "few_shot_random":
    # random: short and about use the same demo games per test game
    ids = res_all.pivot(index="appid", columns="variant", values="demo_ids")
    assert (ids["short"].apply(tuple) == ids["about"].apply(tuple)).all()
    print("short and about use the same demonstration games")

if TECHNIQUE == "few_shot_knn":
    # similarity of the closest pool neighbour per game; > 0.95 hints at near-duplicates
    games_idx = games.set_index("appid")
    for level in DESCRIPTION_LEVELS:
        top1 = []
        for a in games["appid"].astype(int):
            name, i = QUERY_ROWS[a]
            sims = EMB[("pool", level)] @ EMB[(name, level)][i]
            j = sims.argmax()
            top1.append(sims[j])
            if sims[j] > 0.95:
                g = games_idx.loc[a]
                print(f"  [{level}] {g['name']} {g['genres']}  <->  "
                      f"{pool_df.iloc[j]['name']} {pool_df.iloc[j]['genres']}  ({sims[j]:.3f})")
        top1 = pd.Series(top1)
        print(level, top1.describe()[["mean", "min", "max"]].round(3).to_dict(),
              "| near-duplicates (>0.95):", (top1 > 0.95).sum())

No evaluated games among the demonstrations
  [short] Hoversteppers: Zone 1 ['Racing', 'Simulation', 'Sports']  <->  Hoversteppers ['Racing', 'Sports']  (1.000)
  [short] Digital Diamond Baseball V12 ['Simulation', 'Sports', 'Strategy']  <->  Digital Diamond Baseball V11 ['Simulation', 'Sports', 'Strategy']  (1.000)
  [short] Psychiatrist Simulator 2: Prologue ['Simulation']  <->  Psychiatrist Simulator 2 ['Simulation']  (0.950)
  [short] Childhood Dreams - Jigsaw Puzzle ['Strategy']  <->  Cat Jigsaw Puzzle Games ['Strategy']  (1.000)
short {'mean': 0.671, 'min': 0.445, 'max': 1.0} | near-duplicates (>0.95): 4
  [about] Digital Diamond Baseball V12 ['Simulation', 'Sports', 'Strategy']  <->  Digital Diamond Baseball V11 ['Simulation', 'Sports', 'Strategy']  (0.996)
about {'mean': 0.647, 'min': 0.486, 'max': 0.996} | near-duplicates (>0.95): 1


# Per-Technique Evaluation

## Parsing
1. Extract the JSON object and read `genres`.
2. Normalise case and whitespace, map spelling variants via `SYNONYMS`.
3. Drop labels outside `CONTENT_GENRES` and log them as Out-of-Vocabulary (OOV).

Rules:
- Unparseable output → empty prediction, counted as an error and not dropped, so all conditions are evaluated on the same games.
- `SYNONYMS` covers spelling variants only. Subgenres are excluded, e.g. JRPG or ARPG do not count as RPG and are OOV.

In [17]:
SYNONYMS = {
    # Action
    "action game": "action",
    "action games": "action",

    # Adventure
    "adventure game": "adventure",
    "adventure games": "adventure",
    "adventures": "adventure",

    # Massively Multiplayer
    "massively-multiplayer": "massively multiplayer",
    "massive multiplayer": "massively multiplayer",
    "massively multiplayer online": "massively multiplayer",
    "mmo": "massively multiplayer",
    "mmog": "massively multiplayer",

    # Racing
    "race": "racing",
    "racing game": "racing",
    "racing games": "racing",

    # RPG
    "role-playing": "rpg",
    "role playing": "rpg",
    "roleplaying": "rpg",
    "role-playing game": "rpg",
    "role playing game": "rpg",
    "roleplaying game": "rpg",
    "rpgs": "rpg",

    # Simulation
    "simulator": "simulation",
    "simulations": "simulation",
    "simulation game": "simulation",
    "sim": "simulation",

    # Sports
    "sport": "sports",
    "sports game": "sports",
    "sports games": "sports",

    # Strategy
    "strategy game": "strategy",
    "strategy games": "strategy",
}

LOOKUP = {g.lower(): g for g in CONTENT_GENRES}

# each target must be a valid label
assert set(SYNONYMS.values()) <= set(LOOKUP), set(SYNONYMS.values()) - set(LOOKUP)

In [18]:
def parse_output(raw):
    """Parses a model output into valid genre labels.

    Args:
        raw: Raw model output.

    Returns:
        Tuple (labels, valid_json, oov): sorted list of valid genres, whether a
        JSON object with a "genres" list was found, and labels outside
        CONTENT_GENRES.
    """
    
    m = re.search(r"\{.*\}", raw or "", re.DOTALL)
    if m is None:
        return [], False, []
    try:
        items = json.loads(m.group())["genres"]
    except (json.JSONDecodeError, KeyError, TypeError):
        return [], False, []
    if not isinstance(items, list):
        return [], False, []

    labels, oov = [], []
    for item in items:
        key = str(item).strip().lower()
        key = SYNONYMS.get(key, key)
        if key in LOOKUP:
            labels.append(LOOKUP[key])
        else:
            oov.append(str(item))
    return sorted(set(labels)), True, oov

In [19]:
parsed = res_all["raw"].apply(parse_output)

res_all["pred"] = parsed.str[0]
res_all["valid_json"] = parsed.str[1]
res_all["oov"] = parsed.str[2]
res_all["gold"] = res_all["gold"].apply(list)

unknown_gold = {g for gs in res_all["gold"] for g in gs} - set(CONTENT_GENRES)
print("Gold labels outside the label list:", unknown_gold or "none")

Gold labels outside the label list: none


In [20]:
# valid_json: share of parseable outputs
# oov_rate: share with ≥1 out-of-list label
# empty_pred: share without any valid label (counted as fully wrong)

parse_report = res_all.groupby("variant").agg(
    n=("raw", "size"),
    valid_json=("valid_json", "mean"),
    oov_rate=("oov", lambda x: (x.str.len() > 0).mean()),
    empty_pred=("pred", lambda x: (x.str.len() == 0).mean()),
)
display(parse_report)

# display all samples with at least one OOV label
display(res_all.loc[res_all["oov"].str.len() > 0, ["appid", "variant", "pred", "oov"]])

,n,valid_json,oov_rate,empty_pred
variant,,,,
about,1009,1.0,0.009911,0.001982
short,1009,1.0,0.004955,0.000000


,appid,variant,pred,oov
627,3100910,about,"[Action, Adventure]",[Puzzle]
809,433890,about,[Adventure],[Puzzle]
915,1014710,about,"[Action, Adventure]",[Music]
1159,1372470,about,[Adventure],"[Horror, Roguelite, Survival]"
1198,593060,short,[Adventure],[Puzzle]
1215,3213560,about,[Action],[Puzzle]
1337,3433180,about,[],[Puzzle]
1353,1786110,about,"[Action, Adventure]",[Puzzle]
1464,2216160,short,[Adventure],[Horror]
1465,2216160,about,[Adventure],[Horror]


In [21]:
for v, g in res_all.groupby("variant"):
    print(v, Counter(l for ls in g["oov"] for l in ls).most_common(10))

about [('Puzzle', 6), ('Horror', 3), ('Music', 1), ('Roguelite', 1), ('Survival', 1)]
short [('Puzzle', 4), ('Horror', 1)]


## Binarisation
Convert the label lists into 0/1 matrices which order is given by CONTENT_GENRES (rows: games sorted by `appid`, columns: `CONTENT_GENRES`).
The binarizer is fitted on the fixed label list so that all conditions share the same columns. Identical row order across conditions is required for the paired bootstrap.

In [22]:
mlb = MultiLabelBinarizer(classes=CONTENT_GENRES)
mlb.fit([CONTENT_GENRES])

Y = {}
for variant, g in res_all.sort_values("appid").groupby("variant"):
    Y[variant] = {
        "appids": g["appid"].to_numpy(),
        "true": mlb.transform(g["gold"]),
        "pred": mlb.transform(g["pred"]),
    }

In [23]:
ref = next(iter(Y.values()))
for variant, d in Y.items():
    assert (d["appids"] == ref["appids"]).all(), f"{variant}: other games/order"
    assert (d["true"] == ref["true"]).all(), f"{variant}: gold is deviating"
    print(variant, "true", d["true"].shape, "pred", d["pred"].shape)

# How often does each genre appear in gold?
support = pd.Series(ref["true"].sum(axis=0), index=mlb.classes_)
print(support)

about true (1009, 8) pred (1009, 8)
short true (1009, 8) pred (1009, 8)
Action                   479
Adventure                460
Massively Multiplayer     25
Racing                    44
RPG                      205
Simulation               249
Sports                    53
Strategy                 233
dtype: int32


In [24]:
# Majority baseline: predicts the two most frequent genres for every game
maj_idx = [list(mlb.classes_).index(g) for g in ["Action", "Adventure"]]
trivial = np.zeros_like(ref["true"])
trivial[:, maj_idx] = 1

Y_eval = {**Y, "majority": {"appids": ref["appids"], "true": ref["true"], "pred": trivial}}

In [25]:
# Labels per game: more predicted than gold labels raises recall at the expense of precision
for variant, d in Y.items():
    print(variant,
          "labels/game gold:", d["true"].sum(axis=1).mean().round(2),
          "pred:", d["pred"].sum(axis=1).mean().round(2),
          "empty predictions:", (d["pred"].sum(axis=1) == 0).sum())

about labels/game gold: 1.73 pred: 1.77 empty predictions: 2
short labels/game gold: 1.73 pred: 1.68 empty predictions: 0


## Metrics
- Micro-F1 (dominated by frequent genres)
- Macro-F1 (each genre weighted equally)
- micro precision/recall
- Hamming loss and subset accuracy (exact match)
- Macro-F1 is averaged over genres with gold support > 0 (`labels=present`), with full support in the evaluation set this has no effect
- `macro avg` row of `classification_report` averages over all genres and is used for error analysis only

In [26]:
def compute_metrics(y_true, y_pred):
    """Computes multi-label metrics.

    Macro-F1 is averaged over genres with gold support > 0.

    Args:
        y_true: Binary gold matrix (games × genres).
        y_pred: Binary prediction matrix of the same shape.

    Returns:
        Dict with micro precision/recall/F1, macro-F1, Hamming loss and subset accuracy.
    """
    
    present = np.where(y_true.sum(axis=0) > 0)[0]  # genres with gold support > 0
    return {
        "micro_p":    precision_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_r":    recall_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_f1":   f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1":   f1_score(y_true, y_pred, average="macro", labels=present, zero_division=0),
        "hamming":    hamming_loss(y_true, y_pred),
        "subset_acc": accuracy_score(y_true, y_pred),
    }

In [27]:
rows = [
    {"technique": TECHNIQUE, "variant": v, "n": len(d["appids"]),
     **compute_metrics(d["true"], d["pred"])}
    for v, d in Y_eval.items()
]
metrics_df = pd.DataFrame(rows).set_index("variant").round(3)
display(metrics_df)

,technique,n,micro_p,micro_r,micro_f1,macro_f1,hamming,subset_acc
variant,,,,,,,,
about,few_shot_knn,1009,0.700,0.715,0.707,0.641,0.128,0.365
short,few_shot_knn,1009,0.691,0.669,0.680,0.616,0.136,0.343
majority,few_shot_knn,1009,0.465,0.537,0.499,0.159,0.234,0.105


In [28]:
# Per-genre report for error analysis only. Its "macro avg" row averages over all genres and can differ from macro_f1 in metrics_df if a genre has no gold support.

for v, d in Y.items():
    print(f"=== {v} ===")
    print(classification_report(d["true"], d["pred"],
                                target_names=list(mlb.classes_), zero_division=0))

=== about ===
                       precision    recall  f1-score   support

               Action       0.79      0.82      0.81       479
            Adventure       0.66      0.81      0.73       460
Massively Multiplayer       0.21      0.60      0.32        25
               Racing       0.67      0.66      0.67        44
                  RPG       0.72      0.65      0.68       205
           Simulation       0.69      0.60      0.65       249
               Sports       0.70      0.62      0.66        53
             Strategy       0.75      0.54      0.62       233

            micro avg       0.70      0.71      0.71      1748
            macro avg       0.65      0.66      0.64      1748
         weighted avg       0.72      0.71      0.71      1748
          samples avg       0.74      0.76      0.71      1748

=== short ===
                       precision    recall  f1-score   support

               Action       0.76      0.76      0.76       479
            Adventure  

In [29]:
# predicted vs. gold label counts per genre which shall reveal systematically over- or under-assigned genres
pred_counts = pd.DataFrame({v: d["pred"].sum(axis=0) for v, d in Y.items()}, index=mlb.classes_)
pred_counts["gold"] = ref["true"].sum(axis=0)
display(pred_counts)

,about,short,gold
Action,497,479,479
Adventure,559,572,460
Massively Multiplayer,70,40,25
Racing,43,38,44
RPG,185,163,205
Simulation,216,211,249
Sports,47,41,53
Strategy,167,149,233


In [30]:
# Sanity check of the computation of Micro-F1
d = Y["short"]
tp = (d["true"] & d["pred"]).sum()
fp = ((1 - d["true"]) & d["pred"]).sum()
fn = (d["true"] & (1 - d["pred"])).sum()
print("manual:", round(2 * tp / (2 * tp + fp + fn), 3), "| sklearn:", metrics_df.loc["short", "micro_f1"])

manual: 0.68 | sklearn: 0.68


### Paired bootstrap comparisons
The paired bootstrap estimates how much a metric varies only because of which games happen to be in the evaluation set. In each round, n games are drawn with replacement from the n evaluated games, and the metrics of all variants are computed on this **same** resample ("paired"). Repeating this B times gives a distribution per metric; its 2.5th and 97.5th percentiles form a 95 % confidence interval.
- **Per variant:** precision of the estimate, e.g. Micro-F1 of `short`.
- **Differences:** if the interval of a difference (e.g. `about − short`) excludes 0, the difference is statistically significant.

Pairing reduces variance because differences in game difficulty affect both variants equally and cancel out.

**Decision (fixed before main results):** Bonferroni correction over all planned comparisons, B = 10,000, applied in the joint evaluation of all techniques.
The intervals in this section (B = 2,000, uncorrected) are preliminary checks per technique; all reported results come from the joint evaluation.

In [31]:
rng = np.random.default_rng(RANDOM_SEED)
B = 2000
METRICS = ["micro_f1", "macro_f1"]

variants = list(Y_eval)
y_true = ref["true"]
n = len(y_true)

boot = {v: {k: np.empty(B) for k in METRICS} for v in variants}
for b in range(B):
    idx = rng.integers(0, n, n)
    for v in variants:
        m = compute_metrics(y_true[idx], Y_eval[v]["pred"][idx])
        for k in METRICS:
            boot[v][k][b] = m[k]

In [32]:
point = {v: compute_metrics(y_true, Y_eval[v]["pred"]) for v in variants}

ci_df = pd.DataFrame([
    {"variant": v, "metric": k, "point": point[v][k],
     "ci_low": np.percentile(boot[v][k], 2.5),
     "ci_high": np.percentile(boot[v][k], 97.5)}
    for v in variants for k in METRICS
])
display(ci_df.round(3))

,variant,metric,point,ci_low,ci_high
0,about,micro_f1,0.707,0.690,0.723
1,about,macro_f1,0.641,0.610,0.668
2,short,micro_f1,0.680,0.663,0.697
3,short,macro_f1,0.616,0.585,0.646
4,majority,micro_f1,0.499,0.478,0.519
5,majority,macro_f1,0.159,0.154,0.164


In [33]:
COMPARISONS = [("about", "short"), ("short", "name_only"), ("about", "name_only"), ("name_only", "majority")]
COMPARISONS = [(a, b) for a, b in COMPARISONS if a in point and b in point]

diff_df = pd.DataFrame([
    {"comparison": f"{a} − {b}", "metric": k,
     "diff": point[a][k] - point[b][k],
     "ci_low": np.percentile(boot[a][k] - boot[b][k], 2.5),
     "ci_high": np.percentile(boot[a][k] - boot[b][k], 97.5)}
    for a, b in COMPARISONS for k in METRICS
])
diff_df["significant"] = (diff_df["ci_low"] > 0) | (diff_df["ci_high"] < 0)
display(diff_df.round(3))

,comparison,metric,diff,ci_low,ci_high,significant
0,about − short,micro_f1,0.027,0.010,0.044,True
1,about − short,macro_f1,0.025,0.001,0.049,True


In [34]:
# mean of the bootstrap values should be close to the point estimator
for v in variants:
    print(v, "Point:", round(point[v]["micro_f1"], 3),
          "| Bootstrap mean:", round(boot[v]["micro_f1"].mean(), 3))

about Point: 0.707 | Bootstrap mean: 0.707
short Point: 0.68 | Bootstrap mean: 0.68
majority Point: 0.499 | Bootstrap mean: 0.499


## Export
Point estimates, intervals and parsing statistics per variant are merged into one table and saved together with the run configuration in `results/{TECHNIQUE}_{RUN}/`:
- `metrics.csv`: metrics, 95 % CIs (B = 2,000), parsing statistics
- `differences.csv`: uncorrected differences between variants
- `per_class.csv`, `pred_counts.csv`: per-genre results for error analysis
- `config.json`: model, prompt, parameters, seeds, n

In [35]:
RUN_NAME = f"{TECHNIQUE}_{RUN}"
OUT = Path("results") / RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)

In [36]:
ci_wide = ci_df.pivot(index="variant", columns="metric", values=["ci_low", "ci_high"])
ci_wide.columns = [f"{m}_{b}" for b, m in ci_wide.columns]  

results = (
    pd.DataFrame(point).T                                    
    .join(ci_wide)
    .join(parse_report[["valid_json", "oov_rate", "empty_pred"]])
)
results.insert(0, "technique", TECHNIQUE)
results.insert(1, "n", n)
results.index.name = "variant"
results["condition"] = results.index.map({"name_only": "control", "majority": "baseline"}).fillna("description")

results = results.loc[[v for v in ORDER if v in results.index]]
display(results.round(3))

,technique,n,micro_p,micro_r,micro_f1,macro_f1,hamming,subset_acc,macro_f1_ci_low,micro_f1_ci_low,macro_f1_ci_high,micro_f1_ci_high,valid_json,oov_rate,empty_pred,condition
variant,,,,,,,,,,,,,,,,
short,few_shot_knn,1009,0.691,0.669,0.680,0.616,0.136,0.343,0.585,0.663,0.646,0.697,1.0,0.005,0.000,description
about,few_shot_knn,1009,0.700,0.715,0.707,0.641,0.128,0.365,0.610,0.690,0.668,0.723,1.0,0.010,0.002,description
majority,few_shot_knn,1009,0.465,0.537,0.499,0.159,0.234,0.105,0.154,0.478,0.164,0.519,NaN,NaN,NaN,baseline


In [37]:
results.to_csv(OUT / "metrics.csv")
diff_df.to_csv(OUT / "differences.csv", index=False)
pred_counts.to_csv(OUT / "pred_counts.csv")

per_class = pd.concat({
    v: pd.DataFrame(classification_report(d["true"], d["pred"], target_names=list(mlb.classes_),
                                          zero_division=0, output_dict=True)).T
    for v, d in Y.items()
})
per_class.to_csv(OUT / "per_class.csv")

config = {
    "split_seed": SPLIT_SEED,
    "eval_split": EVAL_SPLIT,
    "run": RUN_NAME,
    "sample_seed": RANDOM_SEED,
    "technique": TECHNIQUE, 
    "model": MODEL,
    "embed_model": EMBED_MODEL if TECHNIQUE == "few_shot_knn" else None,
    "system_prompt": SYSTEM_PROMPT,
    "fields": {"description": "Description", "name_only": "Game title"},
    "temperature": MODEL_TEMP, 
    "max_tokens": MAX_TOKENS, 
    "response_format": RESPONSE_FORMAT,
    "labels": CONTENT_GENRES, 
    "synonyms": SYNONYMS,
    "n_games": int(n), 
    "bootstrap_B": B, 
    "bootstrap_seed": RANDOM_SEED,
    "k": K if TECHNIQUE != "zero_shot" else 0,
    "pool_size": POOL_SIZE if TECHNIQUE != "zero_shot" else None,
    "demo_format": "user/assistant turns, gold genres as JSON",
}
_ = (OUT / "config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False))

In [38]:
# Check, all files written?
print(sorted(p.name for p in OUT.iterdir()))

# Check, values identical after reloading?
reloaded = pd.read_csv(OUT / "metrics.csv", index_col="variant")
print("Identical:", np.allclose(reloaded["micro_f1"], results["micro_f1"]))

# Check, config readable?
print(json.loads((OUT / "config.json").read_text())["model"])

['.ipynb_checkpoints', 'config.json', 'differences.csv', 'metrics.csv', 'per_class.csv', 'pred_counts.csv']
Identical: True
mistral-small-3-2-24b-instruct-2506


## Error Analysis
Shows which errors the model makes, from four perspectives:
- **Per genre:** false positives and false negatives.
- **Per game:** error type (exact, over, under, partial, wrong, empty).
- **Confusions:** with which gold genres false positives co-occur.
- **Qualitative:** a sample of error cases with their description.

Note: Steam genres are assigned by developers and may be incomplete. Some apparent errors may be errors in the gold labels.

In [39]:
# FP and FN per genre: many FP = genre assigned too generously, many FN = genre missed
rows = []
for v, d in Y.items():
    t, p = d["true"].astype(bool), d["pred"].astype(bool)
    for j, g in enumerate(mlb.classes_):
        rows.append({
            "variant": v, "genre": g,
            "support": t[:, j].sum(),
            "tp": (t[:, j] & p[:, j]).sum(),
            "fp": (~t[:, j] & p[:, j]).sum(),
            "fn": (t[:, j] & ~p[:, j]).sum(),
        })
err_class = pd.DataFrame(rows)

display(err_class.pivot(index="genre", columns="variant", values=["fp", "fn"]))

fp          fn      
variant               about short about short
genre                                        
Action                  104   113    86   113
Adventure               188   204    89    92
Massively Multiplayer    55    29    10    14
RPG                      52    55    72    97
Racing                   14    14    15    20
Simulation               66    69    99   107
Sports                   14     8    20    20
Strategy                 42    31   108   115

In [40]:
# Error type per game: shows whether over-, under- or misclassification dominates
def error_type(gold, pred):
    """Classifies the error of one prediction.

    Args:
        gold: Gold genres.
        pred: Predicted genres.

    Returns:
        One of "empty", "exact", "over", "under", "partial", "wrong".
    """
    
    g, p = set(gold), set(pred)
    if not p:      return "empty" # no valid label predicted
    if g == p:     return "exact" # exact match
    if g < p:      return "over" # all gold labels plus extra labels
    if p < g:      return "under" # only correct labels, but not all
    if g & p:      return "partial" # some correct, some wrong
    return "wrong" # no overlap

res_all["error_type"] = res_all.apply(lambda r: error_type(r["gold"], r["pred"]), axis=1)
res_all["n_gold"] = res_all["gold"].str.len()
res_all["n_pred"] = res_all["pred"].str.len()

display(pd.crosstab(res_all["variant"], res_all["error_type"]))
display(res_all.groupby("variant")[["n_gold", "n_pred"]].mean().round(2))

error_type,empty,exact,over,partial,under,wrong
variant,,,,,,
about,2,368,251,76,214,98
short,0,346,219,92,235,117


,n_gold,n_pred
variant,,
about,1.73,1.77
short,1.73,1.68


In [41]:
# Error types per game and variant; cross-table short vs. about
per_game = res_all.pivot(index="appid", columns="variant", values="error_type")[[v for v in ORDER if v in Y]]
per_game["gold"] = res_all.groupby("appid")["gold"].first()
display(per_game)
if "name_only" in per_game:
    only_name = ((per_game["name_only"] == "exact") &
                 (per_game["short"] != "exact") & (per_game["about"] != "exact")).sum()
    print("name_only exact, both descriptions not:", only_name)
display(pd.crosstab(per_game["short"], per_game["about"]))

variant,short,about,gold
appid,,,
39660,over,under,"[Simulation, Strategy]"
57620,over,over,[Strategy]
63000,under,over,"[Action, Strategy]"
201490,exact,exact,"[Simulation, Strategy]"
207750,over,exact,[Action]
...,...,...,...
3480360,over,wrong,[Simulation]
3492060,wrong,wrong,[RPG]
3507030,exact,exact,[Action]


about,empty,exact,over,partial,under,wrong
short,,,,,,
exact,0,235,64,5,30,12
over,0,68,119,5,8,19
partial,0,5,20,34,24,9
under,0,45,17,20,149,4
wrong,2,15,31,12,3,54


In [42]:
# Co-occurrence of false positives: gold genre (row) vs. falsely predicted genre (column); games with several gold genres count in each row
for v, d in Y.items():
    t, p = d["true"].astype(int), d["pred"].astype(int)
    fp = (1 - t) * p # false-positive cells only
    co = pd.DataFrame(t.T @ fp, index=mlb.classes_, columns=mlb.classes_)
    co = co.loc[co.sum(axis=1) > 0, co.sum(axis=0) > 0]
    print(f"=== {v} === (row: gold genre, column: falsely predicted genre)")
    display(co)

=== about === (row: gold genre, column: falsely predicted genre)


,Action,Adventure,Massively Multiplayer,Racing,RPG,Simulation,Sports,Strategy
Action,0,80,41,10,24,26,6,19
Adventure,61,0,17,4,30,32,6,18
Massively Multiplayer,3,1,0,0,3,3,0,2
Racing,6,6,5,0,1,5,3,2
RPG,25,51,3,0,0,10,1,8
Simulation,24,59,11,6,15,0,4,16
Sports,5,4,6,2,2,3,0,2
Strategy,25,44,13,1,13,22,5,0


=== short === (row: gold genre, column: falsely predicted genre)


,Action,Adventure,Massively Multiplayer,Racing,RPG,Simulation,Sports,Strategy
Action,0,99,19,10,22,32,4,17
Adventure,60,0,7,5,35,36,4,7
Massively Multiplayer,3,1,0,0,2,2,0,1
Racing,6,4,1,0,2,7,1,1
RPG,26,52,4,2,0,10,1,3
Simulation,26,50,8,4,21,0,0,11
Sports,4,3,4,0,2,6,0,1
Strategy,36,52,9,2,17,14,3,0


In [43]:
# Random sample of non-exact cases with their text for qualitative error analysis
cases = res_all[res_all["error_type"] != "exact"].merge(
    games[["appid", "name"] + list(DESCRIPTION_LEVELS.values())], on="appid"
)

sample = cases.sample(min(30, len(cases)), random_state=RANDOM_SEED)
for _, r in sample.sort_values(["appid", "variant"]).iterrows():
    text = r["name"] if r["variant"] == "name_only" else r[DESCRIPTION_LEVELS[r["variant"]]]
    print(f"[{r['appid']} | {r['variant']} | {r['error_type']}]")
    print(f"  Gold: {r['gold']}  |  Pred: {r['pred']}  |  OOV: {r['oov']}")
    print(f"  Text: {str(text)[:300]}...\n")

[292600 | short | over]
  Gold: ['Action']  |  Pred: ['Action', 'Adventure']  |  OOV: []
  Text: Elegant and minimal, Duet asks players simply to navigate two orbiting spheres past an increasingly complex maze of falling obstacles. As the difficulty increases, the tension mounts, and a peculiar narrative unfolds....

[345330 | about | partial]
  Gold: ['Action', 'Adventure', 'Strategy']  |  Pred: ['Action', 'Adventure', 'Massively Multiplayer']  |  OOV: []
  Text: Eden Rising is a brand new action-packed Open World game! Explore the vast world of Eden and defend your bases from hordes of alien monsters. Make use of everything you collect to craft powerful defenses, upgrade your weapons, and unlock challenges. Explore a vast open world and uncover the secrets ...

[379640 | short | over]
  Gold: ['Action']  |  Pred: ['Action', 'Adventure']  |  OOV: []
  Text: Bloo Kid 2 is a classic 2D retro-style platformer experience with lovely designed pixel-graphics and a full chiptune soundtrack. 

In [44]:
# TP+FP and TP+FN must match pred_counts
for v in Y:
    e = err_class[err_class["variant"] == v].set_index("genre")
    assert ((e["tp"] + e["fp"]) == pred_counts[v]).all(), f"{v}: TP+FP != predictions"
    assert ((e["tp"] + e["fn"]) == pred_counts["gold"]).all(), f"{v}: TP+FN != gold"
print("Consistent with pred_counts")

# share of exact matches must equal subset accuracy
for v in Y:
    exact_share = (res_all.loc[res_all["variant"] == v, "error_type"] == "exact").mean()
    assert np.isclose(exact_share, point[v]["subset_acc"]), f"{v}: exact ≠ subset_acc"
print("Error types consistent with subset accuracy")

# exactly one error type per game and variant
assert res_all.groupby("variant")["error_type"].count().eq(n).all()
print("One error type per game and variant")

Consistent with pred_counts
Error types consistent with subset accuracy
One error type per game and variant


## Exploratory Analysis
Not part of the planned comparisons. Performance by leak status (genre named verbatim in the description): per variant, and about vs. short on the same games.

In [45]:
# Does the text mention ‘multiplayer’ in connection with incorrect MMO predictions?

j = list(mlb.classes_).index("Massively Multiplayer")
d = Y["about"]
fp_ids = d["appids"][(d["pred"][:, j] == 1) & (d["true"][:, j] == 0)]

kw = r"multiplayer|online|co-?op|pvp"
texts = games.set_index("appid")["about_the_game"]
print("MM-FP:", len(fp_ids))
if len(fp_ids):
    print("share with multiplayer keywords:", round(texts.loc[fp_ids].str.contains(kw, case=False).mean(), 2))
print("share in the whole evaluation set:", round(texts.str.contains(kw, case=False).mean(), 2))

MM-FP: 55
share with multiplayer keywords: 0.78
share in the whole evaluation set: 0.17


In [46]:
leak = games.set_index("appid")
for v, col in [("short", "leak_short_description"), ("about", "leak_about_the_game")]:
    d = Y[v]
    mask = leak.loc[d["appids"], col].to_numpy()
    for flag in (True, False):
        m = compute_metrics(d["true"][mask == flag], d["pred"][mask == flag])
        print(f"{v:6} leak={flag!s:5} n={(mask == flag).sum():4}  "
              f"micro_f1={m['micro_f1']:.3f}  macro_f1={m['macro_f1']:.3f}")

short  leak=True  n= 295  micro_f1=0.760  macro_f1=0.714
short  leak=False n= 714  micro_f1=0.642  macro_f1=0.545
about  leak=True  n= 441  micro_f1=0.750  macro_f1=0.702
about  leak=False n= 568  micro_f1=0.668  macro_f1=0.556


In [47]:
lk = games.set_index("appid").loc[Y["short"]["appids"], ["leak_short_description", "leak_about_the_game"]]
for (ls, la), grp in lk.groupby(["leak_short_description", "leak_about_the_game"]):
    m = lk.index.isin(grp.index)
    fs = compute_metrics(Y["short"]["true"][m], Y["short"]["pred"][m])["micro_f1"]
    fa = compute_metrics(Y["about"]["true"][m], Y["about"]["pred"][m])["micro_f1"]
    print(f"leak short={ls!s:5} about={la!s:5} n={m.sum():4}  "
          f"short={fs:.3f}  about={fa:.3f}  diff={fa - fs:+.3f}")

leak short=False about=False n= 519  short=0.645  about=0.657  diff=+0.012
leak short=False about=True  n= 195  short=0.634  about=0.717  diff=+0.083
leak short=True  about=False n=  49  short=0.753  about=0.763  diff=+0.011
leak short=True  about=True  n= 246  short=0.761  about=0.775  diff=+0.014


# Joint evaluation (all techniques, test set)
All conditions are evaluated on the same 1,009 test games. Paired bootstrap with B = 10,000
(same resamples for all conditions). 
- Per-condition intervals: 95 %.
- Differences: Bonferroni-corrected over 12 planned comparisons × 2 metrics (m = 24), i.e. percentiles 0.104 / 99.896.

In [48]:
JOINT_TECHNIQUES = {
    "zero_shot": ["short", "about", "name_only"],
    "few_shot_random": ["short", "about"],
    "few_shot_knn": ["short", "about"],
}

YJ = {}
for tech, tech_variants in JOINT_TECHNIQUES.items():
    raw = pd.read_json(RAW_DIR / f"{tech}_test.jsonl", lines=True).sort_values("appid")
    raw["pred"] = raw["raw"].apply(lambda r: parse_output(r)[0])
    for v in tech_variants:
        g = raw[raw["variant"] == v]
        YJ[f"{tech}|{v}"] = {
            "appids": g["appid"].to_numpy(),
            "true": mlb.transform(g["gold"].apply(list)),
            "pred": mlb.transform(g["pred"]),
        }

# all conditions: same games, same order, same gold
refj = YJ["zero_shot|short"]
for k, d in YJ.items():
    assert len(d["appids"]) == len(test_df), f"{k}: {len(d['appids'])} games"
    assert (d["appids"] == refj["appids"]).all(), f"{k}: other games/order"
    assert (d["true"] == refj["true"]).all(), f"{k}: gold deviates"

# majority baseline: Action + Adventure for every game
maj_cols = [list(mlb.classes_).index(g) for g in ["Action", "Adventure"]]
maj_pred = np.zeros_like(refj["true"])
maj_pred[:, maj_cols] = 1
YJ["majority"] = {"appids": refj["appids"], "true": refj["true"], "pred": maj_pred}

print(list(YJ))

['zero_shot|short', 'zero_shot|about', 'zero_shot|name_only', 'few_shot_random|short', 'few_shot_random|about', 'few_shot_knn|short', 'few_shot_knn|about', 'majority']


In [49]:
def fast_f1(t, p):
    """Computes micro- and macro-F1 from counts (fast version for bootstrapping).

    Same definitions as compute_metrics.

    Args:
        t: Binary gold matrix.
        p: Binary prediction matrix.

    Returns:
        Tuple (micro_f1, macro_f1).
    """
    
    t, p = t.astype(bool), p.astype(bool)
    tp = (t & p).sum(0); fp = (~t & p).sum(0); fn = (t & ~p).sum(0)
    micro = 2 * tp.sum() / (2 * tp.sum() + fp.sum() + fn.sum())
    denom = 2 * tp + fp + fn
    f1c = np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0)
    return micro, f1c[t.sum(0) > 0].mean()

# must match sklearn exactly
for k, d in YJ.items():
    mi, ma = fast_f1(d["true"], d["pred"])
    ref_m = compute_metrics(d["true"], d["pred"])
    assert np.isclose(mi, ref_m["micro_f1"]) and np.isclose(ma, ref_m["macro_f1"]), k
print("fast_f1 matches compute_metrics")

fast_f1 matches compute_metrics


In [50]:
B_JOINT = 10_000
rng_j = np.random.default_rng(RANDOM_SEED)
joint_conds = list(YJ)
n_j = len(refj["appids"])

boot_j = {c: {"micro_f1": np.empty(B_JOINT), "macro_f1": np.empty(B_JOINT)} for c in joint_conds}
t0 = time.time()
for b in range(B_JOINT):
    idx_b = rng_j.integers(0, n_j, n_j)
    t_b = refj["true"][idx_b]
    for c in joint_conds:
        boot_j[c]["micro_f1"][b], boot_j[c]["macro_f1"][b] = fast_f1(t_b, YJ[c]["pred"][idx_b])
print(f"Bootstrap: {time.time() - t0:.0f} s")

Bootstrap: 12 s


In [51]:
LABEL = {"zero_shot": "Zero-shot", "few_shot_random": "Few-shot random", "few_shot_knn": "Few-shot kNN"}
VARIANT_LABEL = {"short": "short description", "about": "about the game", "name_only": "title only"}

def pretty(c):
    """Returns a readable label for a condition key such as "zero_shot|short"."""
    
    if c == "majority":
        return "Majority baseline"
    t, v = c.split("|")
    return f"{LABEL[t]} ({VARIANT_LABEL[v]})"

rows = []
for c in joint_conds:
    m = compute_metrics(YJ[c]["true"], YJ[c]["pred"])
    row = {"condition": c, "label": pretty(c), **m,
           "labels_per_game": YJ[c]["pred"].sum(1).mean()}
    for k in ["micro_f1", "macro_f1"]:
        row[f"{k}_ci_low"], row[f"{k}_ci_high"] = np.percentile(boot_j[c][k], [2.5, 97.5])
    rows.append(row)
joint_results = pd.DataFrame(rows).set_index("condition")
display(joint_results.round(3))

,label,micro_p,micro_r,micro_f1,macro_f1,hamming,subset_acc,labels_per_game,micro_f1_ci_low,micro_f1_ci_high,macro_f1_ci_low,macro_f1_ci_high
condition,,,,,,,,,,,,
zero_shot|short,Zero-shot (short description),0.681,0.646,0.663,0.583,0.142,0.317,1.642,0.646,0.680,0.552,0.610
zero_shot|about,Zero-shot (about the game),0.655,0.735,0.693,0.618,0.141,0.323,1.944,0.676,0.709,0.591,0.643
zero_shot|name_only,Zero-shot (title only),0.577,0.565,0.571,0.482,0.184,0.237,1.697,0.550,0.591,0.450,0.510
few_shot_random|short,Few-shot random (short description),0.701,0.630,0.663,0.598,0.138,0.335,1.557,0.645,0.681,0.567,0.625
few_shot_random|about,Few-shot random (about the game),0.719,0.689,0.704,0.641,0.126,0.369,1.659,0.687,0.720,0.613,0.665
few_shot_knn|short,Few-shot kNN (short description),0.691,0.669,0.680,0.616,0.136,0.343,1.678,0.663,0.697,0.583,0.645
few_shot_knn|about,Few-shot kNN (about the game),0.700,0.715,0.707,0.641,0.128,0.365,1.768,0.690,0.724,0.611,0.668
majority,Majority baseline,0.465,0.537,0.499,0.159,0.234,0.105,2.000,0.478,0.519,0.154,0.164


In [52]:
COMPARISONS_JOINT = [
    # (b) ICL technique, per description level
    ("technique", "few_shot_random|short", "zero_shot|short"),
    ("technique", "few_shot_random|about", "zero_shot|about"),
    ("technique", "few_shot_knn|short",    "zero_shot|short"),
    ("technique", "few_shot_knn|about",    "zero_shot|about"),
    ("technique", "few_shot_knn|short",    "few_shot_random|short"),
    ("technique", "few_shot_knn|about",    "few_shot_random|about"),
    # (a) description scope, per technique
    ("scope", "zero_shot|about",       "zero_shot|short"),
    ("scope", "few_shot_random|about", "few_shot_random|short"),
    ("scope", "few_shot_knn|about",    "few_shot_knn|short"),
    # control conditions
    ("control", "zero_shot|short",     "zero_shot|name_only"),
    ("control", "zero_shot|about",     "zero_shot|name_only"),
    ("control", "zero_shot|name_only", "majority"),
]

ALPHA = 0.05
M_TESTS = len(COMPARISONS_JOINT) * 2                 # 12 comparisons × 2 metrics
q_lo, q_hi = 100 * ALPHA / (2 * M_TESTS), 100 - 100 * ALPHA / (2 * M_TESTS)
print(f"m = {M_TESTS}, percentiles {q_lo:.3f} / {q_hi:.3f}")

rows = []
for group, a, b in COMPARISONS_JOINT:
    for k in ["micro_f1", "macro_f1"]:
        diffs = boot_j[a][k] - boot_j[b][k]
        lo, hi = np.percentile(diffs, [q_lo, q_hi])
        rows.append({"group": group, "comparison": f"{pretty(a)} − {pretty(b)}", "metric": k,
                     "diff": joint_results.loc[a, k] - joint_results.loc[b, k],
                     "ci_low": lo, "ci_high": hi, "significant": (lo > 0) | (hi < 0)})
joint_diff = pd.DataFrame(rows)
display(joint_diff.round(3))

m = 24, percentiles 0.104 / 99.896


,group,comparison,metric,diff,ci_low,ci_high,significant
0,technique,Few-shot random (short description) − Zero-shot (short description),micro_f1,0.000,-0.018,0.019,False
1,technique,Few-shot random (short description) − Zero-shot (short description),macro_f1,0.015,-0.009,0.043,False
2,technique,Few-shot random (about the game) − Zero-shot (about the game),micro_f1,0.011,-0.009,0.032,False
3,technique,Few-shot random (about the game) − Zero-shot (about the game),macro_f1,0.023,-0.005,0.050,False
4,technique,Few-shot kNN (short description) − Zero-shot (short description),micro_f1,0.017,-0.003,0.037,False
5,technique,Few-shot kNN (short description) − Zero-shot (short description),macro_f1,0.033,0.002,0.067,True
6,technique,Few-shot kNN (about the game) − Zero-shot (about the game),micro_f1,0.014,-0.009,0.038,False
7,technique,Few-shot kNN (about the game) − Zero-shot (about the game),macro_f1,0.023,-0.013,0.059,False
8,technique,Few-shot kNN (short description) − Few-shot random (short description),micro_f1,0.017,-0.005,0.038,False
9,technique,Few-shot kNN (short description) − Few-shot random (short description),macro_f1,0.018,-0.017,0.052,False


In [53]:
# Create tables for the essay

def f3(x):
    """Formats a number with three decimals and no leading zero (APA style)."""
    return f"{x:.3f}".replace("0.", ".")
def fci(p, lo, hi):
    """Formats a point estimate with its confidence interval, e.g. ".693 [.676, .709]"."""
    return f"{f3(p)} [{f3(lo)}, {f3(hi)}]"
def fdiff(d, lo, hi):
    """Formats a signed difference with its confidence interval, e.g. "+.030 [+.008, +.053]"."""
    return f"{d:+.3f} [{lo:+.3f}, {hi:+.3f}]".replace("0.", ".")

paper_main = pd.DataFrame({
    "Micro-F1 [95% CI]": [fci(r.micro_f1, r.micro_f1_ci_low, r.micro_f1_ci_high) for r in joint_results.itertuples()],
    "Macro-F1 [95% CI]": [fci(r.macro_f1, r.macro_f1_ci_low, r.macro_f1_ci_high) for r in joint_results.itertuples()],
    "Precision": joint_results["micro_p"].map(f3).to_numpy(),
    "Recall": joint_results["micro_r"].map(f3).to_numpy(),
    "Labels/game": joint_results["labels_per_game"].map(lambda x: f"{x:.2f}").to_numpy(),
}, index=joint_results["label"])

order = [(g, f"{pretty(a)} − {pretty(b)}") for g, a, b in COMPARISONS_JOINT]
paper_diff = (joint_diff
              .assign(cell=[fdiff(r.diff, r.ci_low, r.ci_high) + (" *" if r.significant else "")
                            for r in joint_diff.itertuples()])
              .pivot(index=["group", "comparison"], columns="metric", values="cell")
              .loc[order, ["micro_f1", "macro_f1"]])

display(paper_main)
display(paper_diff)

OUT_J = Path("results") / "joint"
OUT_J.mkdir(parents=True, exist_ok=True)
joint_results.to_csv(OUT_J / "metrics.csv")
joint_diff.to_csv(OUT_J / "differences.csv", index=False)
paper_main.to_csv(OUT_J / "paper_table_main.csv")
paper_diff.to_csv(OUT_J / "paper_table_differences.csv")
_ = (OUT_J / "config.json").write_text(json.dumps({
    "raw_files": [f"{t}_test.jsonl" for t in JOINT_TECHNIQUES],
    "n_games": int(n_j), "bootstrap_B": B_JOINT, "bootstrap_seed": RANDOM_SEED,
    "alpha": ALPHA, "m_tests": M_TESTS, "percentiles": [q_lo, q_hi],
    "comparisons": [[g, a, b] for g, a, b in COMPARISONS_JOINT],
}, indent=2))
print(sorted(p.name for p in OUT_J.iterdir()))

,Micro-F1 [95% CI],Macro-F1 [95% CI],Precision,Recall,Labels/game
label,,,,,
Zero-shot (short description),".663 [.646, .680]",".583 [.552, .610]",.681,.646,1.64
Zero-shot (about the game),".693 [.676, .709]",".618 [.591, .643]",.655,.735,1.94
Zero-shot (title only),".571 [.550, .591]",".482 [.450, .510]",.577,.565,1.70
Few-shot random (short description),".663 [.645, .681]",".598 [.567, .625]",.701,.630,1.56
Few-shot random (about the game),".704 [.687, .720]",".641 [.613, .665]",.719,.689,1.66
Few-shot kNN (short description),".680 [.663, .697]",".616 [.583, .645]",.691,.669,1.68
Few-shot kNN (about the game),".707 [.690, .724]",".641 [.611, .668]",.700,.715,1.77
Majority baseline,".499 [.478, .519]",".159 [.154, .164]",.465,.537,2.00


metric                                                                                          micro_f1  \
group     comparison                                                                                       
technique Few-shot random (short description) − Zero-shot (short description)       +.000 [-.018, +.019]   
          Few-shot random (about the game) − Zero-shot (about the game)             +.011 [-.009, +.032]   
          Few-shot kNN (short description) − Zero-shot (short description)          +.017 [-.003, +.037]   
          Few-shot kNN (about the game) − Zero-shot (about the game)                +.014 [-.009, +.038]   
          Few-shot kNN (short description) − Few-shot random (short description)    +.017 [-.005, +.038]   
          Few-shot kNN (about the game) − Few-shot random (about the game)          +.004 [-.019, +.027]   
scope     Zero-shot (about the game) − Zero-shot (short description)              +.030 [+.008, +.053] *   
          Few-shot random (about the game) − Few-shot random (short description)  +.040 [+.016, +.064] *   
          Few-shot kNN (about the game) − Few-shot kNN (short description)        +.027 [+.001, +.053] *   
control   Zero-shot (short description) − Zero-shot (title only)                  +.093 [+.061, +.127] *   
          Zero-shot (about the game) − Zero-shot (title only)                     +.122 [+.092, +.154] *   
          Zero-shot (title only) − Majority baseline                              +.072 [+.037, +.107] *   

metric                                                                                          macro_f1  
group     comparison                                                                                      
technique Few-shot random (short description) − Zero-shot (short description)       +.015 [-.009, +.043]  
          Few-shot random (about the game) − Zero-shot (about the game)             +.023 [-.005, +.050]  
          Few-shot kNN (short description) − Zero-shot (short description)        +.033 [+.002, +.067] *  
          Few-shot kNN (about the game) − Zero-shot (about the game)                +.023 [-.013, +.059]  
          Few-shot kNN (short description) − Few-shot random (short description)    +.018 [-.017, +.052]  
          Few-shot kNN (about the game) − Few-shot random (about the game)          +.000 [-.034, +.038]  
scope     Zero-shot (about the game) − Zero-shot (short description)              +.035 [+.003, +.071] *  
          Few-shot random (about the game) − Few-shot random (short description)  +.043 [+.004, +.080] *  
          Few-shot kNN (about the game) − Few-shot kNN (short description)          +.025 [-.010, +.063]  
control   Zero-shot (short description) − Zero-shot (title only)                  +.101 [+.048, +.152] *  
          Zero-shot (about the game) − Zero-shot (title only)                     +.136 [+.088, +.186] *  
          Zero-shot (title only) − Majority baseline                              +.323 [+.275, +.369] *

['config.json', 'differences.csv', 'metrics.csv', 'paper_table_differences.csv', 'paper_table_main.csv']


In [54]:
# Sanity checks

# Point estimates must match the per-technique runs
for tech in JOINT_TECHNIQUES:
    per_run = pd.read_csv(Path("results") / f"{tech}_main" / "metrics.csv", index_col="variant")
    for v in JOINT_TECHNIQUES[tech]:
        assert np.isclose(per_run.loc[v, "micro_f1"], joint_results.loc[f"{tech}|{v}", "micro_f1"]), (tech, v)
print("Point estimates identical to per-technique runs")

# Each point estimate lies inside its own interval
assert ((joint_results["micro_f1"] >= joint_results["micro_f1_ci_low"]) &
        (joint_results["micro_f1"] <= joint_results["micro_f1_ci_high"])).all()
print("Point estimates inside their intervals")

Point estimates identical to per-technique runs
Point estimates inside their intervals
